# 日中変動率

cron で出力された HTML のテーブルを、note 用に KaTeX の表形式へ変換

In [5]:
from html.parser import HTMLParser

from pylatexenc.latexencode import unicode_to_latex

name_html = "report/2026/up_down_ratio_10-27.html"


class SimpleTableParser(HTMLParser):

    def __init__(self):
        super().__init__()
        self.table_data = []  # テーブル全体のデータを格納（2次元配列）
        self.current_row = []  # 現在処理中の行データを格納
        self.in_cell = False  # tdまたはthの内部にいるかどうかのフラグ
        self.cell_text = ""  # セル内のテキストを一時保存
        self.latex_left = r"\tiny{\texttt{"
        self.latex_right = r"}}"

    def handle_starttag(self, tag, attrs):
        # trタグの開始で新しい行のリストを初期化
        if tag == "tr":
            self.current_row = []
        # tdまたはthタグの開始でセル内フラグを立て、テキストを初期化
        elif tag in ("td", "th"):
            self.in_cell = True
            self.cell_text = ""

    def handle_data(self, data):
        # セル内部のテキストであれば蓄積する（前後の余白はトリム）
        if self.in_cell:
            self.cell_text += data

    def handle_endtag(self, tag):
        # tdまたはthの終了で、蓄積したテキストを行リストに追加
        if tag in ("td", "th"):
            self.in_cell = False
            self.current_row.append(
                f"{self.latex_left}{unicode_to_latex(self.cell_text.strip(), unknown_char_warning=False,)}{self.latex_right}"
            )
        # trの終了で、完成した行をテーブルデータに追加
        elif tag == "tr":
            if self.current_row:  # 空行でなければ追加
                self.table_data.append(self.current_row)


def to_note_katex(text):
    # 1. '\\' (改行) が '\' になってしまう対策 ──> '\\\\' に増やす
    # ※Pythonの文字列内なので、表記上はエスケープして4本になります
    text = text.replace("\\\\", "\\\\\\\\")

    # 2. '\#' が '#' になってしまう対策 ──> '\\#' に増やす
    text = text.replace("\\#", "\\\\#")

    # 3. '\&' が '&' になってしまう対策 ──> '\\&' に増やす
    text = text.replace("\\&", "\\\\&")

    # 4. \ensuremath{\bigtriangleup} ──> △
    text = text.replace(r"\ensuremath{\bigtriangleup}", "△")

    return text


# open関数でテキストファイルとして読み込む
with open(name_html, "r", encoding="utf-8") as f:
    html_content = f.read()

# パースの実行
parser = SimpleTableParser()
parser.feed(html_content)

# 結果の表示
print(r"$$")
print(r"\begin{array}{r|l|l|c|l|r|r|r|r|c}")
print(r"\hline")
for row in parser.table_data:
    print(to_note_katex(" & ".join(row) + r" \\"))
    print(r"\hline")

print(r"\end{array}")
print(r"$$")

$$
\begin{array}{r|l|l|c|l|r|r|r|r|c}
\hline
\tiny{\texttt{\\#}} & \tiny{\texttt{コード}} & \tiny{\texttt{銘柄名}} & \tiny{\texttt{区分}} & \tiny{\texttt{33業種区分}} & \tiny{\texttt{高値}} & \tiny{\texttt{安値}} & \tiny{\texttt{変動率}} & \tiny{\texttt{出来高}} & \tiny{\texttt{増減}} \\\\
\hline
\tiny{\texttt{1}} & \tiny{\texttt{7356}} & \tiny{\texttt{Retty}} & \tiny{\texttt{G}} & \tiny{\texttt{サービス業}} & \tiny{\texttt{201}} & \tiny{\texttt{126}} & \tiny{\texttt{0.549}} & \tiny{\texttt{13,689,800}} & \tiny{\texttt{▼}} \\\\
\hline
\tiny{\texttt{2}} & \tiny{\texttt{5216}} & \tiny{\texttt{倉元製作所}} & \tiny{\texttt{S}} & \tiny{\texttt{ガラス・土石製品}} & \tiny{\texttt{160}} & \tiny{\texttt{124}} & \tiny{\texttt{0.287}} & \tiny{\texttt{3,396,600}} & \tiny{\texttt{△}} \\\\
\hline
\tiny{\texttt{3}} & \tiny{\texttt{4260}} & \tiny{\texttt{ハイブリッドテクノロジーズ}} & \tiny{\texttt{G}} & \tiny{\texttt{情報・通信業}} & \tiny{\texttt{365}} & \tiny{\texttt{275}} & \tiny{\texttt{0.278}} & \tiny{\texttt{5,096,200}} & \tiny{\texttt{△}} \\\\
\hline
\t